# Module 4: Open Quantum Systems, Dissipation & Trajectories
## Notebook 4.1: Density Matrix, Purity & Entanglement Entropy

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qutip/computational-quantum-optics/blob/main/module_4_open_systems_dissipation/01_density_matrix_and_purity.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/qutip/computational-quantum-optics)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-brightgreen?logo=python)](https://www.python.org/)

---

### Computational Quantum Optics Curriculum
* **Module 0:** Mathematical & Quantum Foundations
* **Module 1:** Single-Mode Quantum States of Light
* **Module 2:** Phase Space Quasiprobability Distributions
* **Module 3:** Light-Matter Interaction & Cavity QED
* **Module 4:** **Open Quantum Systems, Dissipation & Trajectories**
  * **4.1 Density Matrix, Purity & Entanglement Entropy** $\leftarrow$ *(You are here)*
  * **4.2** The Lindblad Master Equation & Dynamical Semigroups
  * **4.3** Cavity Decay, Photon Loss & Thermal Reservoirs
  * **4.4** The Monte Carlo Wavefunction (Quantum Jump) Method
  * **4.5** Quantum Trajectories & Homodyne Detection Unraveling
  * **4.6** Input-Output Theory & Cavity Scattering
  * **4.7** Laser Theory, Micromaser & Trapping States

---

### Learning Objectives
1. **The Density Operator Formalism:** Define the density matrix $\hat{\rho}$ for pure and mixed quantum states, establishing Hermiticity, unit trace, and positive semidefiniteness.
2. **Superposition vs. Statistical Mixture:** Differentiate quantum coherences (off-diagonal elements) from classical probability distributions (diagonal populations).
3. **Purity and Entropy Measures:** Quantify quantum mixedness using purity $\gamma = \mathrm{Tr}(\hat{\rho}^2)$, linear entropy $S_L$, and von Neumann entropy $S(\hat{\rho}) = -\mathrm{Tr}(\hat{\rho}\ln\hat{\rho})$.
4. **The Partial Trace & Reduced Density Operators:** Calculate the partial trace $\hat{\rho}_A = \mathrm{Tr}_B(\hat{\rho}_{AB})$ and evaluate bipartite entanglement entropy.
5. **Bloch Ball Geometry for Qubits:** Map $2 \times 2$ density matrices to the 3D unit ball, linking purity $\gamma = (1 + |\vec{r}|^2)/2$ to the Bloch radius.


In [ ]:
# Environment Setup: Silent Colab detection & high-DPI publication styling
import os
import sys

# Configure headless Agg backend to ensure bulletproof plotting across all environments
if "MPLBACKEND" not in os.environ:
    os.environ["MPLBACKEND"] = "Agg"

import math
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt

# Configure publication matplotlib styling
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "lines.linewidth": 2.0,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})

try:
    import qutip as qt
    HAS_QUTIP = True
    print("QuTiP successfully imported.")
except ImportError:
    HAS_QUTIP = False
    print("QuTiP not found. Using native NumPy/SciPy implementations.")

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found. Interactive exploration will run in static fallback mode.")


---

## 1. Physical Motivation: Pure States vs. Statistical Mixtures

In closed quantum systems, a state is described by a single state vector $|\psi\rangle$ residing in a Hilbert space $\mathcal{H}$. However, real physical systems interact with their environment, suffer from classical experimental uncertainties, or belong to entangled composite systems. Such systems cannot be described by a state vector; they require the **density operator** $\hat{\rho}$.

An ensemble of pure states $\{|\psi_i\rangle\}$ prepared with classical probabilities $\{p_i\}$ ($p_i \ge 0, \sum_i p_i = 1$) is represented by:
$$\hat{\rho} = \sum_{i} p_i |\psi_i\rangle\langle\psi_i|$$

### Fundamental Properties of Any Physical Density Operator:
1. **Hermiticity:** $\hat{\rho}^\dagger = \hat{\rho}$ (all eigenvalues $\lambda_k$ are real).
2. **Unit Trace:** $\mathrm{Tr}(\hat{\rho}) = \sum_k \lambda_k = 1$ (conservation of total probability).
3. **Positive Semidefiniteness:** $\langle \phi | \hat{\rho} | \phi \rangle \ge 0 \quad \forall |\phi\rangle \iff \lambda_k \ge 0 \quad \forall k$.

### Quantum Coherent Superposition vs. Incoherent Mixture
Consider a two-level system with basis $\{|0\rangle, |1\rangle\}$:
* **Coherent Superposition ($|\psi\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}$):**
  $$\hat{\rho}_{\text{coh}} = |\psi\rangle\langle\psi| = \begin{pmatrix} 1/2 & 1/2 \\ 1/2 & 1/2 \end{pmatrix}$$
  The non-zero off-diagonal elements $\rho_{01} = \rho_{10} = 1/2$ represent **quantum coherences**, enabling quantum interference.
* **Classical Statistical Mixture (50% $|0\rangle$, 50% $|1\rangle$):**
  $$\hat{\rho}_{\text{mix}} = \frac{1}{2}|0\rangle\langle 0| + \frac{1}{2}|1\rangle\langle 1| = \begin{pmatrix} 1/2 & 0 \\ 0 & 1/2 \end{pmatrix}$$
  The off-diagonal coherences are identically zero. This is a purely classical mixture of ignorance.

---

## 2. Quantitative Measures of Mixedness & Entropy

How do we measure how "pure" or "mixed" a quantum state is?

### 1. Quantum State Purity ($\gamma$)
$$\gamma(\hat{\rho}) = \mathrm{Tr}(\hat{\rho}^2) = \sum_k \lambda_k^2$$
In a $d$-dimensional Hilbert space:
* For any **pure state** ($\hat{\rho} = |\psi\rangle\langle\psi|$): $\hat{\rho}^2 = \hat{\rho} \implies \gamma = 1.0$.
* For a **mixed state**: $\gamma < 1.0$.
* For the **maximally mixed state** ($\hat{\rho} = \frac{1}{d}\hat{I}_d$): $\gamma = \frac{1}{d}$.
$$\frac{1}{d} \le \gamma(\hat{\rho}) \le 1.0$$

### 2. Linear Entropy ($S_L$)
A normalized measure of mixedness directly related to purity:
$$S_L(\hat{\rho}) = \frac{d}{d-1} \left( 1 - \mathrm{Tr}(\hat{\rho}^2) \right)$$
$S_L = 0$ for pure states, and $S_L = 1.0$ for maximally mixed states.

### 3. Von Neumann Entropy ($S$)
The quantum generalization of Gibbs/Shannon entropy:
$$S(\hat{\rho}) = -\mathrm{Tr}(\hat{\rho} \ln \hat{\rho}) = -\sum_{k} \lambda_k \ln \lambda_k$$
(with the convention $0 \ln 0 \equiv 0$).
* $S(\hat{\rho}) = 0$ if and only if $\hat{\rho}$ is a pure state.
* $S(\hat{\rho}) = \ln d$ for the maximally mixed state.

---

## 3. The Partial Trace & Bipartite Entanglement

Consider a composite quantum system $AB$ living in the tensor product space $\mathcal{H}_A \otimes \mathcal{H}_B$, with density matrix $\hat{\rho}_{AB}$.
The state of subsystem $A$ alone is obtained by "tracing out" (averaging over) all degrees of freedom of subsystem $B$:
$$\hat{\rho}_A = \mathrm{Tr}_B(\hat{\rho}_{AB}) = \sum_{j} (\hat{I}_A \otimes \langle j_B|) \, \hat{\rho}_{AB} \, (\hat{I}_A \otimes |j_B\rangle)$$
where $\{|j_B\rangle\}$ is an orthonormal basis of $\mathcal{H}_B$.

### Entanglement Entropy as an Unambiguous Witness:
For an overall **pure bipartite state** $|\psi_{AB}\rangle$:
1. If $|\psi_{AB}\rangle = |\phi_A\rangle \otimes |\chi_B\rangle$ is **separable (product state)**:
   $$\hat{\rho}_A = |\phi_A\rangle\langle\phi_A| \implies S(\hat{\rho}_A) = 0, \quad \gamma(\hat{\rho}_A) = 1.0$$
2. If $|\psi_{AB}\rangle$ is **entangled**:
   $$\hat{\rho}_A \text{ is strictly mixed} \implies S(\hat{\rho}_A) > 0, \quad \gamma(\hat{\rho}_A) < 1.0$$
The von Neumann entropy of the reduced density matrix $S(\hat{\rho}_A) = S(\hat{\rho}_B)$ is the gold standard measure of pure-state bipartite entanglement (**Entanglement Entropy**)!


In [ ]:
# Implementation: Density Matrix Validator, Partial Trace, Purity & Entropy Functions

class QuantumStateAnalyzer:
    """
    Comprehensive toolset for density matrix operations, partial trace,
    purity, linear entropy, von Neumann entropy, and Bloch vector extraction.
    """
    @staticmethod
    def is_valid_density_matrix(rho, tol=1e-8):
        """Checks Hermiticity, unit trace, and positive semidefiniteness."""
        # 1. Hermiticity: rho == rho^\dagger
        if not np.allclose(rho, rho.conj().T, atol=tol):
            return False, "Not Hermitian"
            
        # 2. Unit Trace: Tr(rho) == 1
        tr = np.trace(rho)
        if abs(tr - 1.0) > tol:
            return False, f"Trace is {tr:.6f}, not 1.0"
            
        # 3. Positive Semidefiniteness: all eigenvalues >= 0
        evals = la.eigvalsh(rho)
        if np.any(evals < -tol):
            return False, f"Negative eigenvalue detected: {np.min(evals):.3e}"
            
        return True, "Valid density matrix"
        
    @staticmethod
    def purity(rho):
        """Computes purity gamma = Tr(rho^2)."""
        return np.real(np.trace(rho @ rho))
        
    @staticmethod
    def von_neumann_entropy(rho, base=np.e):
        """Computes von Neumann entropy S(rho) = -sum(lambda_i * log(lambda_i))."""
        evals = la.eigvalsh(rho)
        evals = np.clip(evals, 0.0, 1.0)
        evals = evals[evals > 1e-15]
        log_vals = np.log(evals) if base == np.e else np.log2(evals)
        return -np.sum(evals * log_vals)
        
    @staticmethod
    def linear_entropy(rho):
        """Computes normalized linear entropy S_L = d/(d-1) * (1 - Tr(rho^2))."""
        d = rho.shape[0]
        if d <= 1:
            return 0.0
        pur = QuantumStateAnalyzer.purity(rho)
        return (float(d) / (d - 1.0)) * (1.0 - pur)
        
    @staticmethod
    def bloch_vector(rho_qubit):
        """Extracts 3D Bloch vector r = (u, v, w) for a 2x2 density matrix."""
        # sigma_x = [[0, 1], [1, 0]]
        # sigma_y = [[0, -1j], [1j, 0]]
        # sigma_z = [[1, 0], [0, -1]]
        u = 2.0 * np.real(rho_qubit[0, 1])
        v = 2.0 * np.imag(rho_qubit[1, 0])
        w = np.real(rho_qubit[0, 0] - rho_qubit[1, 1])
        return np.array([u, v, w])
        
    @staticmethod
    def partial_trace_b(rho_ab, dim_a, dim_b):
        """
        Traces out subsystem B from composite bipartite density matrix rho_AB (dim_A x dim_B).
        Returns reduced density matrix rho_A of dimension dim_A x dim_A.
        """
        # Reshape into a 4-tensor (i_A, i_B, j_A, j_B)
        tensor = rho_ab.reshape(dim_a, dim_b, dim_a, dim_b)
        # Contract over indices of subsystem B (axes 1 and 3)
        rho_a = np.trace(tensor, axis1=1, axis2=3)
        return rho_a

# Verification Benchmark: Analyze Pure, Mixed, and Entangled States
analyzer = QuantumStateAnalyzer()

# 1. Pure superposition qubit
psi_plus = np.array([1.0, 1.0], dtype=complex) / np.sqrt(2.0)
rho_pure = np.outer(psi_plus, psi_plus.conj())

# 2. Maximally mixed qubit
rho_mixed = 0.5 * np.eye(2, dtype=complex)

# 3. Maximally entangled Bell state |Phi+> = (|00> + |11>) / sqrt(2)
psi_bell = np.zeros(4, dtype=complex)
psi_bell[0] = 1.0 / np.sqrt(2.0)  # |00>
psi_bell[3] = 1.0 / np.sqrt(2.0)  # |11>
rho_bell = np.outer(psi_bell, psi_bell.conj())

# Trace out subsystem B
rho_a_reduced = analyzer.partial_trace_b(rho_bell, dim_a=2, dim_b=2)

print("--- Quantum State Analyzer Benchmark ---")
print(f"Pure State Purity:             {analyzer.purity(rho_pure):.6f} (Expected 1.0)")
print(f"Pure State Entropy:            {analyzer.von_neumann_entropy(rho_pure):.6f} (Expected 0.0)")
print(f"Maximally Mixed State Purity:  {analyzer.purity(rho_mixed):.6f} (Expected 0.5)")
print(f"Maximally Mixed State Entropy: {analyzer.von_neumann_entropy(rho_mixed):.6f} (Expected ln(2) = {np.log(2):.6f})")
print(f"Bell Reduced State Purity:     {analyzer.purity(rho_a_reduced):.6f} (Expected 0.5)")
print(f"Bell Entanglement Entropy:     {analyzer.von_neumann_entropy(rho_a_reduced):.6f} (Expected ln(2) = {np.log(2):.6f})")

assert abs(analyzer.purity(rho_pure) - 1.0) < 1e-12, "Pure state purity check failed!"
assert abs(analyzer.von_neumann_entropy(rho_pure)) < 1e-12, "Pure state entropy check failed!"
assert abs(analyzer.purity(rho_mixed) - 0.5) < 1e-12, "Mixed state purity check failed!"
assert abs(analyzer.purity(rho_a_reduced) - 0.5) < 1e-12, "Bell reduced state check failed!"
print("Verification Passed: All state validation, partial trace, and entropy metrics match theoretical bounds!")


In [ ]:
# Publication-Quality 4-Panel Visualization: Superposition vs. Mixture, Bloch Ball, Entanglement & Entropy Bound

fig = plt.figure(figsize=(16, 12))

# =============================================================================
# Panel 1: Density Matrix Heatmaps: Coherent Superposition vs. Incoherent Mixture
# =============================================================================
ax1_a = fig.add_subplot(2, 4, 1)
ax1_b = fig.add_subplot(2, 4, 2)

im1 = ax1_a.imshow(np.real(rho_pure), cmap='coolwarm', vmin=-0.1, vmax=0.6)
ax1_a.set_title(r"Superposition: $|\psi\rangle = \frac{|0\rangle+|1\rangle}{\sqrt{2}}$" + "\n" + r"$\gamma = 1.0$, $S = 0$")
ax1_a.set_xticks([0, 1])
ax1_a.set_yticks([0, 1])
ax1_a.set_xticklabels([r"$|0\rangle$", r"$|1\rangle$"])
ax1_a.set_yticklabels([r"$|0\rangle$", r"$|1\rangle$"])
for i in range(2):
    for j in range(2):
        ax1_a.text(j, i, f"{np.real(rho_pure[i, j]):.2f}", ha='center', va='center', fontsize=11, fontweight='bold')

im2 = ax1_b.imshow(np.real(rho_mixed), cmap='coolwarm', vmin=-0.1, vmax=0.6)
ax1_b.set_title(r"Classical Mixture: $\frac{1}{2}|0\rangle\langle 0| + \frac{1}{2}|1\rangle\langle 1|$" + "\n" + r"$\gamma = 0.5$, $S = \ln 2$")
ax1_b.set_xticks([0, 1])
ax1_b.set_yticks([0, 1])
ax1_b.set_xticklabels([r"$|0\rangle$", r"$|1\rangle$"])
ax1_b.set_yticklabels([r"$|0\rangle$", r"$|1\rangle$"])
for i in range(2):
    for j in range(2):
        ax1_b.text(j, i, f"{np.real(rho_mixed[i, j]):.2f}", ha='center', va='center', fontsize=11, fontweight='bold')

# =============================================================================
# Panel 2: Bloch Ball Cross-Section: Purity Contours from Surface to Center
# =============================================================================
ax2 = fig.add_subplot(2, 2, 2)

theta_circle = np.linspace(0, 2*np.pi, 300)
# Outer pure boundary r = 1.0
ax2.plot(np.cos(theta_circle), np.sin(theta_circle), 'k-', lw=2.0, label=r"Pure Surface $r = 1$ ($\gamma = 1.0$)")

# Radial concentric purity contours r = [0.25, 0.5, 0.75]
radii = [0.25, 0.5, 0.75]
contour_colors = ['#1f77b4', '#2ca02c', '#ff7f0e']
for r_val, col in zip(radii, contour_colors):
    pur_val = 0.5 * (1.0 + r_val**2)
    ax2.plot(r_val * np.cos(theta_circle), r_val * np.sin(theta_circle), color=col, linestyle='--', lw=1.5,
             label=rf"$r = {r_val:.2f}$ ($\gamma = {pur_val:.3f}$)")

# Central maximally mixed point
ax2.plot([0], [0], 'r*', markersize=14, label=r"Center $r = 0$ ($\hat{\rho} = \frac{1}{2}\hat{I}$, $\gamma = 0.5$)")

ax2.set_xlim(-1.18, 1.18)
ax2.set_ylim(-1.18, 1.18)
ax2.set_aspect('equal')
ax2.set_xlabel(r"Bloch Component $u = \mathrm{Tr}(\hat{\rho}\hat{\sigma}_x)$")
ax2.set_ylabel(r"Bloch Component $w = \mathrm{Tr}(\hat{\rho}\hat{\sigma}_z)$")
ax2.set_title(r"Bloch Ball Geometry: $\gamma = \frac{1 + |\vec{r}|^2}{2}$")
ax2.legend(loc="upper right", fontsize=8.5, frameon=True)

# =============================================================================
# Panel 3: Bipartite Entanglement: |psi(theta)> = cos(theta)|00> + sin(theta)|11>
# =============================================================================
ax3 = fig.add_subplot(2, 2, 3)

theta_scan = np.linspace(0.0, np.pi / 2.0, 400)
purity_reduced = []
entropy_reduced = []

for th in theta_scan:
    psi_th = np.zeros(4, dtype=complex)
    psi_th[0] = np.cos(th)  # |00>
    psi_th[3] = np.sin(th)  # |11>
    rho_th = np.outer(psi_th, psi_th.conj())
    rho_a = QuantumStateAnalyzer.partial_trace_b(rho_th, 2, 2)
    
    purity_reduced.append(QuantumStateAnalyzer.purity(rho_a))
    entropy_reduced.append(QuantumStateAnalyzer.von_neumann_entropy(rho_a) / np.log(2.0))

ax3.plot(theta_scan / np.pi, purity_reduced, 'b-', lw=2.2, label=r"Subsystem Purity $\gamma(\hat{\rho}_A)$")
ax3.plot(theta_scan / np.pi, entropy_reduced, 'purple', lw=2.2, label=r"Entanglement Entropy $S(\hat{\rho}_A)$ [Bits]")

ax3.axvline(0.25, color='red', linestyle=':', lw=1.8, label=r"Maximal Bell State $\theta = \pi/4$")
ax3.axhline(0.5, color='gray', linestyle=':', alpha=0.6)
ax3.axhline(1.0, color='gray', linestyle=':', alpha=0.6)

ax3.set_xlabel(r"State Parameter $\theta / \pi$")
ax3.set_ylabel("Metric Value")
ax3.set_title(r"Entanglement vs. Local Mixedness: $|\psi\rangle = \cos\theta|00\rangle + \sin\theta|11\rangle$")
ax3.set_ylim(-0.02, 1.08)
ax3.legend(loc="center right", frameon=True)

# =============================================================================
# Panel 4: Purity vs. Von Neumann Entropy for Random Mixed Qubit States
# =============================================================================
ax4 = fig.add_subplot(2, 2, 4)

# Generate an ensemble of 1500 random physical mixed qubit states
np.random.seed(42)
pur_samples = []
ent_samples = []

for _ in range(1500):
    # Ginibre random matrix A
    A = np.random.randn(2, 2) + 1j * np.random.randn(2, 2)
    rho_rand = A @ A.conj().T
    rho_rand /= np.trace(rho_rand)
    
    pur_samples.append(QuantumStateAnalyzer.purity(rho_rand))
    ent_samples.append(QuantumStateAnalyzer.von_neumann_entropy(rho_rand))

# Exact analytical boundary for qubits: lambda1 = (1 + r)/2, lambda2 = (1 - r)/2
# gamma = (1 + r^2)/2 -> r = sqrt(2*gamma - 1)
gamma_curve = np.linspace(0.5, 1.0, 300)
r_curve = np.sqrt(2.0 * gamma_curve - 1.0)
lam1 = 0.5 * (1.0 + r_curve)
lam2 = 0.5 * (1.0 - r_curve)
# Handle lam2=0
ent_boundary = -(lam1 * np.log(np.clip(lam1, 1e-12, 1.0)) + np.where(lam2 > 1e-12, lam2 * np.log(np.clip(lam2, 1e-12, 1.0)), 0.0))

ax4.scatter(pur_samples, ent_samples, color='dodgerblue', alpha=0.3, s=12, label="Random Qubit States")
ax4.plot(gamma_curve, ent_boundary, 'r-', lw=2.5, label="Universal Qubit Bound")

ax4.set_xlabel(r"State Purity $\gamma = \mathrm{Tr}(\hat{\rho}^2)$")
ax4.set_ylabel(r"Von Neumann Entropy $S(\hat{\rho})$ [Nats]")
ax4.set_title("Universal Purity vs. Entropy Relationship")
ax4.set_xlim(0.48, 1.02)
ax4.set_ylim(-0.02, 0.74)
ax4.legend(loc="upper right", frameon=True)

plt.tight_layout()
plt.show()


> **Physical Insight: The GHJW Theorem & The Indeterminacy of Quantum Ensembles**
>
> In classical statistical mechanics, if you are told that a coin has a 50% probability of being Heads and 50% Tails, the preparation is unambiguous: the system was prepared in one of two distinct physical states.
>
> In quantum mechanics, however, consider the maximally mixed qubit state $\hat{\rho} = \frac{1}{2}\hat{I}_2$:
> 1. It can be prepared by a source flipping a classical coin and sending $|0\rangle$ or $|1\rangle$ along the $z$-axis:
>    $$\hat{\rho} = \frac{1}{2}|0\rangle\langle 0| + \frac{1}{2}|1\rangle\langle 1| = \begin{pmatrix} 1/2 & 0 \\ 0 & 1/2 \end{pmatrix}$$
> 2. It can equally well be prepared by sending $|+\rangle$ or $|-\rangle$ along the $x$-axis:
>    $$\hat{\rho} = \frac{1}{2}|+\rangle\langle +| + \frac{1}{2}|-\rangle\langle -| = \begin{pmatrix} 1/2 & 0 \\ 0 & 1/2 \end{pmatrix}$$
> 3. It can be produced by preparing an entangled Bell pair $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$ and throwing away subsystem $B$!
>
> **The GHJW (Gisin-Hughston-Jozsa-Wootters) Theorem** proves that *no measurement allowed by quantum mechanics can ever distinguish between these completely different preparations*. The density operator $\hat{\rho}$ contains all information accessible in the physical universe, showing that quantum states are fundamentally defined by observable expectation values rather than classical ontological histories.


In [ ]:
# Interactive Exploration: Bloch Vector, Decoherence & Mixedness Control

def plot_interactive_density(r_bloch=0.8, theta_deg=45.0):
    """
    Simulates a general qubit density matrix rho = (I + r*sigma) / 2
    and calculates purity, linear entropy, and von Neumann entropy.
    """
    th_rad = np.radians(theta_deg)
    u = r_bloch * np.sin(th_rad)
    w = r_bloch * np.cos(th_rad)
    v = 0.0
    
    # rho = 0.5 * [[1 + w, u - 1j*v], [u + 1j*v, 1 - w]]
    rho = 0.5 * np.array([[1.0 + w, u - 1j*v], [u + 1j*v, 1.0 - w]], dtype=complex)
    
    pur = QuantumStateAnalyzer.purity(rho)
    ent_vn = QuantumStateAnalyzer.von_neumann_entropy(rho)
    lin_ent = QuantumStateAnalyzer.linear_entropy(rho)
    
    fig, (ax_bloch, ax_bars) = plt.subplots(1, 2, figsize=(14, 4.5))
    
    # Bloch disk cross-section
    circ_angles = np.linspace(0, 2*np.pi, 200)
    ax_bloch.plot(np.cos(circ_angles), np.sin(circ_angles), 'k-', lw=1.8, label="Unit Sphere (Pure States)")
    ax_bloch.plot(0.5 * np.cos(circ_angles), 0.5 * np.sin(circ_angles), 'gray', linestyle=':', label=r"Half-Radius $r = 0.5$")
    
    # Plot state vector
    ax_bloch.arrow(0, 0, u, w, head_width=0.06, head_length=0.06, fc='crimson', ec='crimson', lw=2.5,
                   label=rf"State Vector $|\vec{{r}}| = {r_bloch:.2f}$")
    ax_bloch.plot([0], [0], 'ko', markersize=6)
    
    ax_bloch.set_xlim(-1.15, 1.15)
    ax_bloch.set_ylim(-1.15, 1.15)
    ax_bloch.set_aspect('equal')
    ax_bloch.set_xlabel(r"$u = \mathrm{Tr}(\hat{\rho}\hat{\sigma}_x)$")
    ax_bloch.set_ylabel(r"$w = \mathrm{Tr}(\hat{\rho}\hat{\sigma}_z)$")
    ax_bloch.set_title(rf"Bloch Vector Position ($\theta = {theta_deg:.0f}^\circ$)")
    ax_bloch.legend(loc="upper right", fontsize=8.5, frameon=True)
    
    # Metrics bar chart
    metrics = ['Purity gamma', 'Linear Entropy S_L', 'Von Neumann S / ln(2)']
    vals = [pur, lin_ent, ent_vn / np.log(2.0)]
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    
    bars = ax_bars.bar(metrics, vals, color=colors, edgecolor='navy', alpha=0.8, width=0.5)
    ax_bars.set_ylim(0.0, 1.15)
    ax_bars.set_ylabel("Normalized Value")
    ax_bars.set_title("Mixedness & Entropy Metrics")
    
    for bar, val in zip(bars, vals):
        ax_bars.text(bar.get_x() + bar.get_width()/2.0, val + 0.03, f"{val:.4f}",
                     ha='center', va='bottom', fontsize=10, fontweight='bold')
                     
    plt.tight_layout()
    plt.show()

if HAS_WIDGETS:
    widgets.interact(
        plot_interactive_density,
        r_bloch=widgets.FloatSlider(value=0.8, min=0.0, max=1.0, step=0.05, description=r"$|\vec{r}|$:"),
        theta_deg=widgets.FloatSlider(value=45.0, min=0.0, max=180.0, step=15.0, description=r"$\theta^\circ$:")
    )
else:
    print("Static exploration demonstration (default parameters: |r| = 0.8, theta = 45 deg):")
    plot_interactive_density(r_bloch=0.8, theta_deg=45.0)


---

## 4. Self-Check Exercises & Guided Challenge

### Challenge 1: The Werner State Purity and Subsystem Entanglement
The Werner state for two qubits ($d=4$) is a mixture of a maximally entangled Bell state $|\Phi^+\rangle$ and complete white noise:
$$\hat{\rho}_W(p) = p |\Phi^+\rangle\langle\Phi^+| + \frac{1-p}{4} \hat{I}_4, \quad p \in [0, 1]$$

1. Derive the global purity $\gamma(\hat{\rho}_W)$ as a function of the mixing fraction $p$.
2. Calculate the reduced state of subsystem $A$ and show that $\hat{\rho}_A = \frac{1}{2}\hat{I}_2$ for all $p$.

<details>
<summary><b>Solution & Derivation</b> (Click to expand)</summary>

1. **Global State Purity:**
   Since $|\Phi^+\rangle\langle\Phi^+|$ is an orthogonal projection operator onto a 1D subspace, and $\hat{I}_4$ is the identity:
   $$\hat{\rho}_W^2 = p^2 |\Phi^+\rangle\langle\Phi^+| + \frac{2p(1-p)}{4} |\Phi^+\rangle\langle\Phi^+| + \frac{(1-p)^2}{16} \hat{I}_4$$
   Taking the trace using $\mathrm{Tr}(|\Phi^+\rangle\langle\Phi^+|) = 1$ and $\mathrm{Tr}(\hat{I}_4) = 4$:
   $$\mathrm{Tr}(\hat{\rho}_W^2) = p^2 + \frac{p(1-p)}{2} + \frac{(1-p)^2}{4} = \frac{4p^2 + 2p - 2p^2 + 1 - 2p + p^2}{4} = \frac{3p^2 + 1}{4}$$
   * At $p = 1$ (pure Bell state): $\gamma = \frac{3(1) + 1}{4} = 1.0$.
   * At $p = 0$ (completely mixed): $\gamma = \frac{1}{4} = 0.25$.

2. **Reduced Subsystem State:**
   $$\hat{\rho}_A = \mathrm{Tr}_B(\hat{\rho}_W) = p \, \mathrm{Tr}_B(|\Phi^+\rangle\langle\Phi^+|) + \frac{1-p}{4} \, \mathrm{Tr}_B(\hat{I}_4)$$
   Since $\mathrm{Tr}_B(|\Phi^+\rangle\langle\Phi^+|) = \frac{1}{2}\hat{I}_2$ and $\mathrm{Tr}_B(\hat{I}_4) = 2\hat{I}_2$:
   $$\hat{\rho}_A = p \left(\frac{1}{2}\hat{I}_2\right) + \frac{1-p}{4} (2\hat{I}_2) = \frac{p}{2}\hat{I}_2 + \frac{1-p}{2}\hat{I}_2 = \frac{1}{2}\hat{I}_2$$
   Subsystem $A$ is **maximally mixed for every value of $p$**!
</details>

---

### Challenge 2: Numerical Verification of Werner State Purity
For mixing fraction $p = 0.6$, construct the Werner state $\hat{\rho}_W$ in Python. Compute the numerical purity $\mathrm{Tr}(\hat{\rho}_W^2)$ and verify that it matches the theoretical prediction $\frac{3(0.6)^2 + 1}{4} = \frac{1.08 + 1}{4} = 0.520000$.


In [ ]:
# Numerical Verification of Challenge 2: Werner State Purity Benchmark

p_val = 0.6
psi_bell = np.zeros(4, dtype=complex)
psi_bell[0] = 1.0 / np.sqrt(2.0)
psi_bell[3] = 1.0 / np.sqrt(2.0)
proj_bell = np.outer(psi_bell, psi_bell.conj())

# Construct Werner state
rho_werner = p_val * proj_bell + ((1.0 - p_val) / 4.0) * np.eye(4, dtype=complex)

# Validate density matrix
is_valid, msg = QuantumStateAnalyzer.is_valid_density_matrix(rho_werner)
assert is_valid, f"Werner state validation failed: {msg}"

# Numerical metrics
pur_num = QuantumStateAnalyzer.purity(rho_werner)
pur_theory = (3.0 * p_val**2 + 1.0) / 4.0

# Reduced state
rho_a_w = QuantumStateAnalyzer.partial_trace_b(rho_werner, 2, 2)
pur_a_w = QuantumStateAnalyzer.purity(rho_a_w)

print(f"--- Werner State Benchmark (p = {p_val}) ---")
print(f"Density Matrix Validity:        {msg}")
print(f"Theoretical Global Purity:      {pur_theory:.6f}")
print(f"Numerical Global Purity:        {pur_num:.6f}")
print(f"Subsystem A Reduced Purity:     {pur_a_w:.6f} (Expected 0.500000)")
print(f"Purity Discrepancy:             {abs(pur_num - pur_theory):.3e}")

assert abs(pur_num - pur_theory) < 1e-12, "Werner purity check failed!"
assert abs(pur_a_w - 0.5) < 1e-12, "Werner subsystem reduced purity check failed!"
print("Verification Passed: Numerical Werner state satisfies algebraic purity formula to machine precision!")


---

## 5. Summary & Key Formulas

| Concept | Mathematical Formula | Physical Meaning |
| :--- | :--- | :--- |
| **Density Matrix** | $\hat{\rho} = \sum_i p_i |\psi_i\rangle\langle\psi_i|$ | Complete statistical description of quantum systems |
| **State Purity** | $\gamma = \mathrm{Tr}(\hat{\rho}^2)$ | Measures mixedness ($1/d \le \gamma \le 1.0$) |
| **Von Neumann Entropy** | $S(\hat{\rho}) = -\mathrm{Tr}(\hat{\rho}\ln\hat{\rho})$ | Information entropy ($0 \le S \le \ln d$) |
| **Bloch Ball Radius** | $r = |\vec{r}| = \sqrt{2\gamma - 1}$ | Qubit geometry: pure on surface ($r=1$), mixed in interior ($r<1$) |
| **Partial Trace** | $\hat{\rho}_A = \mathrm{Tr}_B(\hat{\rho}_{AB})$ | Reduced state of subsystem; $S(\hat{\rho}_A)$ measures entanglement |

---

### Further Reading & Foundational Literature
* **J. von Neumann**, *Mathematical Foundations of Quantum Mechanics*, Princeton University Press (1955).
* **M. A. Nielsen & I. L. Chuang**, *Quantum Computation and Quantum Information*, Cambridge University Press (2010), Chapter 2.
* **H. P. Breuer & F. Petruccione**, *The Theory of Open Quantum Systems*, Oxford University Press (2002), Chapter 2.

---

### Next Notebook in Module 4
In **Notebook 4.2: The Lindblad Master Equation & Dynamical Semigroups**, we analyze how the density matrix evolves in time under open dissipative dynamics:
* Microscopic derivation from system-reservoir coupling via Born-Markov and secular approximations.
* The Gorini-Kossakowski-Sudarshan-Lindblad (GKSL) theorem.
* Constructing jump operators $\hat{L}_k$ and the Lindblad dissipator $\mathcal{D}[\hat{L}]\hat{\rho}$.
* Spontaneous decay, thermal excitation, and pure dephasing superoperators.
